# New 3-step SFT generation: single-hop target 800

New 3-step pipeline SFT generation. Fill `API_KEY`, `BASE_URL`, and `MODEL`, then run the code cell.


In [ ]:
import os
import shlex
import shutil
import subprocess
import sys
import time
from pathlib import Path

PROJECT = Path("/root/autodl-tmp/visual_rag_agent")
os.chdir(PROJECT)
PYTHON = sys.executable

# Fill these before running. Do not print the API key.
API_KEY = ""
BASE_URL = ""  # OpenAI-compatible endpoint, e.g. https://.../v1
MODEL = ""     # e.g. qwen3.6-plus-2026-04-02 / qwen3.6-max-preview / mimo-v2.5
CONCURRENCY = 8

# Validator stays DeepSeek. Generation uses the API/model above.
VALIDATOR_MODEL = "deepseek-v4-flash"
HOP = "single"
TARGET_KEPT = 800
CANDIDATE_ROWS = 1200
RUN_ID = time.strftime("new3step_single800_%Y%m%d_%H%M%S")

assert API_KEY.strip(), "Fill API_KEY before running."
assert BASE_URL.strip(), "Fill BASE_URL before running."
assert MODEL.strip(), "Fill MODEL before running."

DATA_ROOT = PROJECT / f"data/corpora/slidevqa_train_{RUN_ID}"
INDEX_DIR = PROJECT / f"data/indexes/slidevqa_train_{RUN_ID}"
OUTPUT_DIR = PROJECT / f"outputs/sft_trajectories/{RUN_ID}/{HOP}"
LOG_DIR = PROJECT / f"logs/sft_generation/{RUN_ID}"
LOG_DIR.mkdir(parents=True, exist_ok=True)

DATASET_FILE = DATA_ROOT / ("train_multi.jsonl" if HOP == "multi" else "train_single.jsonl")

print("RUN_ID:", RUN_ID)
print("DATASET_FILE:", DATASET_FILE)
print("INDEX_DIR:", INDEX_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)


def run(cmd, *, env=None):
    shown = " ".join(shlex.quote(str(x)) for x in cmd)
    print("\n$", shown, flush=True)
    p = subprocess.Popen(
        [str(x) for x in cmd],
        cwd=PROJECT,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert p.stdout is not None
    for line in p.stdout:
        print(line, end="", flush=True)
    code = p.wait()
    if code != 0:
        raise subprocess.CalledProcessError(code, cmd)

# 1) Prepare a fresh train split subset for this hop type.
shutil.rmtree(INDEX_DIR, ignore_errors=True)
prepare_cmd = [
    PYTHON, "scripts/prepare_slidevqa_train_balanced.py",
    "--parquet-root", "/root/autodl-tmp/hf_data/NTT-hil-insight-SlideVQA/data",
    "--output-root", str(DATA_ROOT),
    "--strategy", "packed",
    "--clean",
]
if HOP == "multi":
    prepare_cmd += ["--single-target", "0", "--multi-target", str(CANDIDATE_ROWS)]
else:
    prepare_cmd += ["--single-target", str(CANDIDATE_ROWS), "--multi-target", "0"]
run(prepare_cmd)

# 2) Build the retriever index over the same page root used by generation.
run([
    PYTHON, "scripts/build_index.py",
    "--dataset", str(DATASET_FILE),
    "--page-root", str(DATA_ROOT / "pages"),
    "--output", str(INDEX_DIR),
])

# 3) Run new 3-step SFT trajectory generation.
#    decide input: search history + current evidence_state + retained highlighted images.
#    analyse output: think/useful_cells/one-sentence summary/judge.
#    evidence_update output: evidence_state={observed_evidence, remaining_gap} for yes/partial only.
env = os.environ.copy()
env["PATH"] = str(Path(PYTHON).parent) + os.pathsep + env.get("PATH", "")
env.update({
    "MIMO_API_KEY": API_KEY,
    "MIMO_BASE_URL": BASE_URL,
    "MAN_SFT_GENERATOR_MODEL": MODEL,
    "DEEPSEEK_MODEL": VALIDATOR_MODEL,
    "SFT_DATASET_FILE": str(DATASET_FILE),
    "SFT_PAGE_ROOT": str(DATA_ROOT / "pages"),
    "SFT_INDEX_DIR": str(INDEX_DIR),
    "SFT_RUN_ID": RUN_ID,
    "SFT_TRAJECTORY_OUTPUT_DIR": str(OUTPUT_DIR),
    "SFT_START_INDEX": "0",
    "SFT_MAX_SAMPLES": str(CANDIDATE_ROWS),
    "SFT_TARGET_KEPT": str(TARGET_KEPT),
    "SFT_CONCURRENCY": str(CONCURRENCY),
    "SFT_MAX_RETRIEVAL_STEPS": "5",
    "SFT_RETRIEVAL_TOP_K": "1",
    "SFT_RETRIEVER_DEVICE": "cuda",
    "SFT_RETRIEVER_DTYPE": "float16",
    "SFT_RETRIEVER_ATTN": "sdpa",
})

run(["bash", "scripts/run_mimo_sft_generation.sh"], env=env)

print("\nDONE")
print("raw:", OUTPUT_DIR / "raw.jsonl")
print("kept:", OUTPUT_DIR / "kept.jsonl")
print("sft calls:", OUTPUT_DIR / "kept_sft_calls.jsonl")
print("summary:", OUTPUT_DIR / "summary.json")
